# Machine Learning on text with TF-IDF and LSA

In [ ]:
!pip install --upgrade 'imbalanced-learn'
import imblearn
import matplotlib.pyplot as plt
import numpy
import pandas
import re
import seaborn
import sklearn.decomposition
import sklearn.ensemble
import sklearn.feature_extraction
import sklearn.linear_model
import sklearn.metrics
import sklearn.model_selection
import sklearn.svm

## Getting data

Execute the cell below to retrieve some textual data.

The dataset is a collection of movie plots and other metadata such as the release year, the director or the genre.

In [ ]:
!git clone https://github.com/nzmonzmp/dataset-wikipedia-movie-plots.git
movie_plot_path = "dataset-wikipedia-movie-plots/wiki_movie_plots_deduped.csv"

## Loading the data in a pandas dataframe

Load the csv file in a dataframe.

Check that the columns are consistent with the csv file. How many movies are there in this dataset?

In [ ]:
# Your code here

### Solution

In [ ]:
df = pandas.read_csv(movie_plot_path)
print(f"Columns: {', '.join(df.columns)}")
print(f"Shape: {df.shape}")

## Selecting data

We will try to predict the genre of a movie using its plot. To do so, we will limit ourselves to the 5 most present genres in the dataset. We will also remove the movies with the most present genre: `unknown`, for obvious reasons.

What are the genres that we will keep?

Build a new dataframe containing only the movies of those genres.

How many movies are left?

Useful functions:

- [`pandas.Series.value_counts`](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.Series.value_counts.html)
- [`pandas.Series.isin`](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.Series.isin.html)

In [ ]:
# Your code here

### Solution

In [ ]:
counts = df["Genre"].value_counts().drop("unknown")
genres = counts.index.values[:5]
print(f"Selected genres: {', '.join(genres)}")
print(f"Number of selected movies: {counts[genres].sum()}")

In [ ]:
clean_df = df[df["Genre"].isin(genres)]
seaborn.countplot(clean_df["Genre"])

## Some preprocessing

A useful preprocessing to reduce overfitting is to change all numbers to a special token.

Use the [`re`](https://docs.python.org/3/library/re.html) module from the standard library to replace all numbers by `aanumber` (any other word not in the dataset would do).

In [ ]:
# Your code here

### Solution

In [ ]:
regex = re.compile(r"[0-9]+")


def preprocess(text: str) -> str:
  return regex.sub("aanumber", text)


x = clean_df["Plot"].map(preprocess)

## TF-IDF representation

Use [`sklearn.feature_extraction.text.TfidfVectorizer`](https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html) to convert plot texts to vectors.

Remove stopwords and keep only the 3000 most present features.

In [ ]:
# Your code here

### Solution

In [ ]:
vectorizer = sklearn.feature_extraction.text.TfidfVectorizer(stop_words='english', max_features=3000)
X = vectorizer.fit_transform(x)
feature_names = vectorizer.get_feature_names()
print(f"10 first columns: {', '.join(feature_names[:10])}")

In [ ]:
indices_0 = X[0].indices
words_0 = [vectorizer.get_feature_names()[i] for i in X[0].indices]
print(f"Columns > 0 in the first line of X : {indices_0}")
print(f"Corresponding words : {', '.join(words_0)}")
print(f"First value of x : {x.iloc[0]}")

## Latent Semantic Analysis

Use [`sklearn.decomposition.TruncatedSVD`](https://scikit-learn.org/stable/modules/generated/sklearn.decomposition.TruncatedSVD.html) to compute LSA and keep only the 5 dimensions that explain the most variance.

Display the eigenvalue of each eigenvector and the quantity of explained variance for each dimension.

In [ ]:
# Your code here

### Solution

In [ ]:
svd = sklearn.decomposition.TruncatedSVD(n_components=5,
                                         n_iter=100,
                                         random_state=42)
svd.fit(X)

print(svd.explained_variance_ratio_)
print(numpy.cumsum(svd.explained_variance_ratio_))
print(svd.singular_values_)

## Important words

Display the 10 words that have the biggest weights in each projection vector by creating a dataframe from them and displaying it.

[`numpy.argsort`](https://numpy.org/doc/stable/reference/generated/numpy.argsort.html) could be useful, or loading in a pandas dataframe.

In [ ]:
# Your code here

### Solution

In [ ]:
def load_components_to_df(components: numpy.ndarray) -> pandas.DataFrame:
  # Get the absolute value of each coeff
  positive = numpy.abs(components)

  # Sort each line and keep the sorted indices
  sorted_indices = numpy.argsort(positive)

  # Get the 10 indices that correspond to the 10 biggest values
  top_10 = sorted_indices[:, :-11:-1]

  # Load the result in a dataframe
  df = pandas.DataFrame(top_10,
                        columns=[f"Word {i + 1}"
                                 for i in range(top_10.shape[1])],
                        index=[f"Topic {i + 1}"
                               for i in range(top_10.shape[0])])

  # Replace each index by the corresponding word.
  df = df.applymap(feature_names.__getitem__)
  return df


load_components_to_df(svd.components_)

## Supervised approach

We will now try to predict the genre of a movie based on its plot.

Start by extracting the genres of the selected movies in a pandas series.

Then, label encode those genres.

In [ ]:
# Your code here

### Solution

In [ ]:
id_to_genre = clean_df["Genre"].unique().tolist()
genre_to_id = {g: i for i, g in enumerate(id_to_genre)}

print("Before:")
display(clean_df["Genre"].value_counts())

y = clean_df["Genre"].map(genre_to_id)

print("After:")
display(y.value_counts())

### Training

Use a classification algorithm to train a model that takes plots tf-idf vectors as input and predicts a genre.

Split the data into train and test sets before training to be able to evaluate your approach.

In [ ]:
# Your code here

### Solution

In [ ]:
def pipeline(model):
  predictions = sklearn.model_selection.cross_val_predict(model, X, y)

  accuracy = sklearn.metrics.accuracy_score(y, predictions)

  micro_precision = sklearn.metrics.precision_score(y,
                                                    predictions,
                                                    average="micro")

  macro_precision = sklearn.metrics.precision_score(y,
                                                    predictions,
                                                    average="macro")

  print(f"Accuracy: {accuracy:.2f}, "
        f"micro precision: {micro_precision:.2f}, "
        f"macro precision: {macro_precision:.2f}")

  confusion_matrix = sklearn.metrics.confusion_matrix(
      y, predictions, normalize="true")
  seaborn.heatmap(confusion_matrix,
                  vmin=0,
                  vmax=1,
                  xticklabels=id_to_genre,
                  yticklabels=id_to_genre,
                  cmap=seaborn.color_palette("Blues", as_cmap=True))
  plt.plot()

In [ ]:
pipeline(sklearn.linear_model.LogisticRegression(penalty="l2", max_iter=200))

In [ ]:
# With under-sampling of most represented genres
pipeline(imblearn.ensemble.BalancedBaggingClassifier(
    base_estimator=sklearn.linear_model.LogisticRegression(penalty="l2",
                                                           max_iter=200)))

In [ ]:
# With over-sampling of least represented genres
pipeline(
    imblearn.pipeline.make_pipeline(
        imblearn.over_sampling.RandomOverSampler(),
        sklearn.linear_model.LogisticRegression(penalty="l2", max_iter=200)
    )
)

In [ ]:
# With a fast SVM
# pipeline(sklearn.svm.LinearSVC(class_weight="balanced"))

In [ ]:
# With a random forest
# pipeline(sklearn.ensemble.RandomForestClassifier(n_estimators=100,
#                                                  class_weight="balanced"))

In [ ]:
# With a slow SVM but with a powerful kernel
# pipeline(sklearn.svm.SVC(class_weight="balanced"))